# COMET InSAR Workshop: LiCSBAS Tutorial — **Module 0: dataset preparation (NISAR data, experimental)***This notebook is the **NISAR data route** of Module 0: data access (ASF search + download of NISAR CSL2 `GUNW` products), conversion to the LiCSBAS `GEOC/` layout, and then the *same* steps `0-2` → `0-5` as in [the LiCSAR tutorial](tutorial_0.ipynb). The time series part is shared: [**`tutorial_1.ipynb`**](tutorial_1.ipynb).*> **What is NISAR?** — NISAR / ALOS-2 is a *dual-frequency (L + P band)* radar satellite. The NISAR CSL2 products distributed by ASF are **interferograms (GUNW)** in `.h5` (HDF5) format, organized into **frames (tracks)**. LiCSBAS uses **GUNW only** — GSLC is not needed.>> **Frames:** a region is usually covered by *several* frames, and each frame is a self-contained time series that must be processed **separately** (do not combine frames; `nisar2geoc` even warns if the input mixes them).>> **Why experimental?** — this chain is new (LiCSBAS v3), and requires the optional extras `licsar_extra[nisar]` + `licsbas[nisar]` (rasterio, rioxarray, dask, xarray, pyproj, h5py, statsmodels; plus `snaphu` for the L-band unwrapping used by the data provider). The steps degrade gracefully (`-h` works everywhere; the heavy imports are lazy).>> Full documentation: [`docs/05_nisar.md`](../docs/05_nisar.md).

### please run the following cell first to load some necessary modules:  
(to run a cell, just click into it and press shift-Enter)  

#### Note: in case your notebook gets frozen/kernel disconnected, try clicking Kernel->Restart (rather than Refreshing page). If this runs ok, you may just rerun the following cell, and then skip back to the cell you were in before.

In [ ]:
import sys, os, subprocess, warnings
import importlib.util as _iu
from IPython.display import Image

def _run(cmd):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if r.returncode != 0:
        print((r.stdout or "")[-3000:])
        raise RuntimeError("command failed: " + cmd)

def _have(mod):
    try:
        return _iu.find_spec(mod) is not None
    except Exception:
        return False

# COMET local jupyterhub: add the legacy bash CLI wrappers (bin/) to PATH, if present
_hub = '/data/notebooks/repositories/LiCSBAS/bin'
if os.path.exists(_hub):
    os.environ['PATH'] = os.environ['PATH'] + ':' + _hub

# ---- LiCSBAS v3 is a pip package ('licsbas'). Make sure the (binary) GDAL stack
# ---- and the package itself are available before anything imports them.
# ---- This is idempotent: on a kernel that already has them, it is a no-op.
_installed = []

# 1) GDAL Python bindings (osgeo) + pyproj/rasterio are the hard binary deps of
#    licsbas. 'osgeo' is NOT on PyPI, so on an environment that has conda
#    (Colab, Binder) we pull it from conda-forge; otherwise it must already be
#    present in the system/conda env.
if not _have("osgeo"):
    if os.path.exists("/opt/conda/bin/conda") or _have("conda"):
        print("GDAL (osgeo) not found - installing from conda-forge (this can take a minute) ...")
        _run("conda install -y -q -c conda-forge gdal pyproj=3.6")
        _installed.append("gdal (conda-forge)")
    else:
        raise SystemExit(
            "GDAL Python bindings ('osgeo') are required by licsbas but not found.\n"
            "Install them, e.g., with:\n"
            "    conda install -c conda-forge gdal pyproj=3.6\n"
            "(on Google Colab this cell should auto-install them via conda-forge).")

# 2) the licsbas package itself
if not _have("licsbas"):
    print("licsbas not found - installing 'licsbas[viewer]' via pip ...")
    _run('pip install -q "licsbas[viewer]"')
    _installed.append("licsbas[viewer]")

warnings.filterwarnings('ignore')

if _installed:
    print("\n[setup] installed: " + ", ".join(_installed))
    print("[setup] New native libraries need a KERNEL RESTART to be picked up.")
    print("[setup]   -> Runtime > Restart session (or Restart kernel), then re-run THIS cell.\n")
    raise SystemExit("Please restart the kernel and re-run this cell to finish setup.")

# some extra functions
def display(png):
    if not os.path.exists(png):
        print('This png file does not exist, recheck please')
    else:
        return Image(png)

def printbold(instr='done'):
    print('\033[1m' + instr + '\033[0m')

# selected frame
frame = '022D_04826_121209'  # for tutorial

# locate the pre-extracted tutorial dataset (binder postBuild / install script)
framedir = None
for cand in (os.getcwd(),
             os.path.join(os.getcwd(), frame),
             os.path.join(os.environ.get('HOME', '.'), frame)):
    if os.path.exists(os.path.join(cand, 'GEOCml2GACOS')):
        framedir = cand
        break

if framedir is None:
    print('WARNING: tutorial dataset not found (GEOCml2GACOS).')
    print('Download and extract it first - see tutorial/install_complete.sh or the binder postBuild script.')
else:
    os.chdir(framedir)

print('prerequisites loaded, thank you. Running in directory:')
print(os.getcwd())
printbold()


## The processing chain of this module (NISAR route)<img src="https://raw.githubusercontent.com/comet-licsar/LiCSBAS/towards-v3/tutorial/0_nisar.png" width="100%">*(image not visible? the PNGs also live in `tutorial/0_nisar.png`; `python3 tutorial/make_flowchart.py` regenerates them)*

## Step 0-1 (NISAR): **search ASF + download NISAR GUNW** — `LiCSBAS01_get_geotiff_NISAR.py`Searches [ASF](https://asf.alaska.edu) for NISAR CSL2 GUNW over a bbox and streams them (via an ASF session, no shell `wget`) into **per-frame subfolders** (the NISAR frame id, e.g. `018A_01234`):```WORKdir/NISAR_GUNW/├── 018A_01234/│   ├── 20250101_20250117.h5│   └── 20250117_20250133.h5└── 019A_01235/...```Useful options (see `-h` for all):- `--list-frames` — show the available frames + how many GUNW per frame, *then exit (no download)*;- `--frame 018A_01234[, 018A_01235, ...]` — download **only** the selected frame(s).> **Prerequisite:** `pip install licsar_extra[nisar]` (the `nisar_data` module with ASF search + download). The `-h` below works even without it.>> **One frame per time series** — each frame folder is its own LiCSBAS dataset (its own `GEOC/`), processed with the steps below and with the time series module.

In [ ]:
![-h help] licsbas-01-get-geotiff-nisar -h

In [ ]:
# EXAMPLES (not run in this tutorial — needs the NISAR deps)# 1) which frames are available here? (no download)# licsbas-01-get-geotiff-nisar -g 14.00/14.25/40.75/40.95 --list-frames# 2) dry-run: what would be downloaded (all frames in the bbox)# licsbas-01-get-geotiff-nisar -g 14.00/14.25/40.75/40.95 -s 20250101 --dry_run# 3) download one frame only# licsbas-01-get-geotiff-nisar -o NISAR_GUNW -g 14.00/14.25/40.75/40.95 --frame 018A_01234 -s 20250101

## NISAR → GEOC conversion — `LiCSBAS_nisar2geoc.py`One **frame folder** is converted to the LiCSBAS `GEOC/<ifg>/...` layout; from `02` onwards the chain is identical to the LiCSAR route:- each IFG: `.geo.unw.tif` / `.geo.cc.tif`, with the smooth **ionospheric ramp** and the **range-offset (pixel-offset) ramp** removed *before* writing (following `earthquake_responder.py`), then the coherence mask (default threshold `0.5`);- top-level files, **written once** from the first GUNW of the frame: `X.geo.E.tif`/`X.geo.N.tif`/`X.geo.U.tif` (from the GUNW's own `radarGrid` metadata) and — with `-D` — `X.geo.hgt.tif` (best-effort NISAR DEM);- the standard **`baselines`** file (per-IFG perpendicular baseline) and a `metadata.txt` (`center_time`, `radar_freq`);- **all GeoTIFFs share one WGS-84 grid** (the first IFG defines it, `rasterio` export; the others are re-registered onto it).> **Do not mix frames (tracks)** — point `-i` at a *single frame* folder (one `GEOC/` per time series). `nisar2geoc` prints a warning if the input GUNWs span more than one frame.

In [ ]:
![-h help] licsbas-nisar2geoc -h

In [ ]:
# EXAMPLES (not run in this tutorial — needs NISAR .h5 input + the NISAR deps)# one frame (track) per GEOC:# licsbas-nisar2geoc -i NISAR_GUNW/018A_01234 -o GEOC_018A_01234 -t 0.5# licsbas-nisar2geoc -i NISAR_GUNW/018A_01234 -o GEOC_018A_01234 -D    # also write hgt (best-effort DEM)# licsbas-nisar2geoc -i NISAR_GUNW/018A_01234 -o GEOC --no_iono --no_offsets --no_deramp_quad --dry_run

## (alternative, GeoTIFF input) — `LiCSBAS_asf2geoc.py``![-h help] licsbas-asf2geoc -h`

## Steps `0-2` → `0-5`: **identical to the LiCSAR route**From here on the chain is exactly the one in [the LiCSAR tutorial](tutorial_0.ipynb) — the same steps, the same flags:

## (skipping) Step 0-2: Convert and downsample

### licsbas-02-ml-prep
This script converts GeoTIFF files of unw and cc to float32 and uint8 format, respectively, for further time series analysis, and also downsamples (multilooks) data if specified.  
Existing files are not re-created to save time, i.e., only the newly available data will be processed.  

**Note:** Use of LiCSBAS02to05_unwrap.py replaces this and following 0-x optional steps, however they still can be applied to its output.

In [ ]:
! licsbas-02-ml-prep -h
# e.g. licsbas-02-ml-prep -i GEOC -o GEOCml10 -n 10 --npara 1
printbold()

# (skipping) Step 0-3: GACOS correction (optional)

**cell \[ 03 \]**

In [ ]:
! licsbas-03op-gacos -h
# e.g. licsbas-03op-gacos -i GEOCml10 -o GEOCml10GACOS --fillhole --n_para 1
printbold()

## (skipping) Step 0-4: Mask UNW (optional)

In [ ]:
! licsbas-04op-mask-unw -h
printbold()

## (skipping) Step 0-5: Clip UNW (optional)

In [ ]:
! licsbas-05op-clip-unw -h
# e.g. licsbas-05op-clip-unw -i GEOCml10GACOS -o GEOCml10GACOSclip -g 14.03/14.22/40.78/40.90 --n_para 1
printbold()

## Next step: Module 1 — Time series (steps `11` → `16` + viewers)The time series analysis (`LiCSBAS11` → `LiCSBAS16`) is **independent of the data source** — continue in [**`tutorial_1.ipynb`**](tutorial_1.ipynb).**[Continue in `tutorial_1.ipynb` (opens in this Binder session) &raquo;](<tutorial_1.ipynb>)**- [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/comet-licsar/LiCSBAS/HEAD?labpath=tutorial/tutorial_1.ipynb) - *new Binder session*- [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/comet-licsar/LiCSBAS/blob/towards-v3/tutorial/tutorial_1.ipynb) - *Colab**Other Module-0 routes: [LiCSAR data](tutorial_0.ipynb) · [SBOVNL/SBOI](tutorial_0_sbovl.ipynb) · more NISAR details in [`docs/05_nisar.md`](../docs/05_nisar.md)*